----
## <font color='CornflowerBlue'>Week 6: Calibration of LLMs</font> 
##### Marieke Smith, Alok Bharadwaj, and Arjen Jakobi
---

Last week, you learnt that language models are effective at capturing syntactic relationships between sequences at arbitrary lengths. This ability makes language models uniquely suited for generating new sequences under various context. In case of human language, one of the most prolific use case is the development of chatbots. 

Language models generate text through auto-regressive sampling of next token. Each  generated token is a prediction drawn from a probability distribution over the entire vocabulary. The chance that a token is picked over others depends not only on its prevalence in its current context in the entire training corpus, but also on the fine-tuning and reinforcement applied to the model after the pre-training stage. 

The mechanism of predicting next-tokens allows a wide range of applications. For creative applications, the generated sequences might be very new and the novelty might even be the desired outcome. However, one of the most used applications of chatbot is for information retreival. In such applications, the generated sequence need to be aligned with objective description of reality. 

- Explain how showing probabilities help to produce an honest output 

In this notebook, we will study the probability of generated tokens on a constrained example. Multiple Choice Questions (MCQs) offer a unique opportunity to study whether  a model is well calibrated to recover factual information. The model can be prompted to provide a structured output which can be parsed and analysed. 

Run the following cell to load the necessary packages. 

In [ ]:
# #@title Click here to initialize the environment (Required) { display-mode: "form" }
import os, sys, subprocess

print("Initializing environment. Please wait.")

# Clone the git repo 
subprocess.run(["git", "clone", "https://github.com/cryoTUD/ai4nanobiology.git"], stdout=subprocess.DEVNULL)
os.chdir("ai4nanobiology/week_6")
sys.path.append(os.path.abspath('.'))

# Installing requirements
subprocess.run(["pip", "install", "-q", "sentencepiece", "protobuf", "tiktoken", "transformers", "accelerate"])
print("Environment ready! You can now start the exercise.")

In [ ]:
import re
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.utils import setup_client

url = setup_client()
print("Proxy URL:", url)

We use the open weight Llama-1b and Llama-8b models. Since these models are large, running inference on local machines are cumbersome. Therefore we use the inference infrastructure provided by **HuggingFace**. Check the function definition in the next code cell

In [ ]:
def query_model(prompt, model_name, max_tokens, client, system_prompt=""):
    """Send one prompt to the proxy and return the model's response.

    Parameters
    ----------
    prompt        : str   the user prompt (the question or batch of questions)
    model_name    : str   "llama-1b", or "llama-8b" 
    max_tokens    : int   how many tokens to generate
    client        : str   proxy URL from setup_client()
    system_prompt : str   optional system instruction

    Returns
    -------
    dict with keys:
        "answer"      : str   the generated text
        "logprobs"    : dict  {token: logprob}
        "token_probs" : dict  {token: probability}
        "logprob_contents" : list of dictionaries for each generated token[{}]
    or None if the request was rate-limited / rejected.
    """
    response = requests.post(
        f"{client}/generate",
        json={
            "system_prompt": system_prompt,
            "prompt": prompt,
            "model": model_name,
            "max_tokens": max_tokens,
        },
        timeout=60,
    )

    if response.status_code == 429:
        print("Rate limit hit - wait a moment and try again.")
        return None
    if response.status_code == 400:
        print(f"Bad request: {response.json().get('detail')}")
        return None

    response.raise_for_status()
    return response.json()

In [ ]:
# Models available through the proxy. Swap MODEL_NAME to compare sizes later.
MODEL_1B = "meta-llama/Llama-3.2-1B-Instruct"
MODEL_8B = "meta-llama/Llama-3.1-8B-Instruct"

MODEL_NAME = MODEL_1B   

# Map the dataset's integer answer (0-3) to a letter (A-D)
ANSWER_MAP = {0: "A", 1: "B", 2: "C", 3: "D"}

## 1. MMLU Dataset
Let us first see how our MCQ dataset looks like. We use the MMLU dataset, which provided questions and answers for several subjects. 

In [ ]:
# Load one MMLU subject (college_biology) so we have a question to play with.
DATASET_NAME = "college_biology"
BASE_URL = "hf://datasets/cais/mmlu/"

def load_subject(dataset_name, split="test"):
    path = f"{dataset_name}/{split}-00000-of-00001.parquet"
    return pd.read_parquet(BASE_URL + path)

df_full = load_subject(DATASET_NAME)
print(f"Loaded {len(df_full)} questions from '{DATASET_NAME}'.")

MMLU stores each question as a row: the `question` text, a list of 4 `choices`,
and an integer `answer` (0-3) giving the correct option.

In [ ]:
df_full.head()

In [ ]:
df_full.iloc[3]

## 2. Querying the model 
Now, let us construct a prompt to feed a question from this dataset into our model. What kind of instruction will you send to the model along with the question? Write your instructions in the code below: 

```python
def build_prompt(question, choices):
    option_lines = [f"{letter}. {text}"
                    for letter, text in zip("ABCD", choices)]
    instruction = "" # <-- Write your instruction so the model only answers with a single letter
    return f"{question}\n" + "\n".join(option_lines) + "\n\n" + instruction
```

In [ ]:
def build_prompt(question, choices):
    option_lines = [f"{letter}. {text}"
                    for letter, text in zip("ABCD", choices)]
    # write your own instruction below in plain English
    instruction =  # TO DO 
    return f"{question}\n" + "\n".join(option_lines) + "\n\n" + instruction

In [ ]:
# Pick one question
q_idx = 3
row = df_full.iloc[q_idx]
correct_letter = ANSWER_MAP[row["answer"]]

prompt = build_prompt(row["question"], row["choices"])

print("-" * 60)
print(prompt)
print("-" * 60)
print(f"Correct answer (from our dataset): {correct_letter}")

Now, ask the model and check the probabilities associated with each returned token

In [ ]:
# Ask the model. We only need a couple of tokens for a single letter.
result = query_model(prompt, MODEL_NAME, max_tokens=5, client=url)

print(f"Model's raw answer: {result['answer']!r}")
print(f"Correct answer    : {correct_letter}")
print()
print("Per-token probabilities the model assigned:")
for tok, p in result["token_probs"].items():
    print(f"  {tok!r:8} -> {p:.4f}")

Ideally, the model followed your instruction and answere with only a single letter (with a nullspace character at the end). If not, try changing your instructions. 

`result["token_probs"]` gives the probability associated with each token in the model output. It is derived directly from the logits that the model outputs. 

## 3. Parsing model response

We need to parse the returned string carefully to capture the true output. We use regex to run parsing. 

In [ ]:
def parse_single_letter(text):
    """Return the first standalone A/B/C/D found in text, or None."""
    match = re.search(r"\b([ABCD])\b", text.strip().upper())
    return match.group(1) if match else None


model_letter = parse_single_letter(result["answer"])
is_correct = (model_letter == correct_letter)

print(f"Parsed model answer: {model_letter}")
print(f"Correct answer     : {correct_letter}")
print(f"Correct?           : {is_correct}")

## 4. Model response on entire dataset

To query answers from many questions, we can loop through all the questions in the dataset and run statistics on them later. However, this runs into practical limits on the number of API requests that can be sent to **Huggingface**. We therefore limit the requests by batching several questions at once and sending a single prompt. For this, we need to ensure that the model responds appropriately for each question, with the right structure. 


In [ ]:
def batch_questions(df, batch_size):
    """Split df into batches; each batch is (prompt_string, {qnum: correct_letter})."""
    prompts, answer_keys = [], []
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]
        header = (
            f"Answer the following {len(batch)} multiple choice questions.\n"
            "For each question, answer with the question number followed by a "
            "letter (A, B, C, or D) and nothing else.\n\n"
        )
        body = ""
        key = {}
        for local_i, (_, row) in enumerate(batch.iterrows(), start=1):
            option_lines = [f"{letter}. {text}"
                            for letter, text in zip("ABCD", row["choices"])]
            body += f"{local_i}: {row['question']}\n" + "\n".join(option_lines) + "\n\n"
            key[local_i] = ANSWER_MAP[row["answer"]]
        prompts.append(header + body)
        answer_keys.append(key)
    return prompts, answer_keys

In [ ]:
BATCH_SIZE = 10
prompts, answer_keys = batch_questions(df_full, batch_size=BATCH_SIZE)
print(f"{len(df_full)} questions -> {len(prompts)} batches of up to {BATCH_SIZE}.")

# Look at one batched prompt
print("\n" + "-" * 60)
print(prompts[0])

A system prompt has a stronger alignment force on the model than the user query. Apply the following system prompt. 


In [ ]:
SYSTEM_PROMPT = """You are an answer key. You receive multiple choice questions and output only the answers.

RULES:
- Output one line per question.
- Each line is exactly: the question number, a colon, a space, and a single capital letter (A, B, C, or D).
- Output nothing else: no explanations, no restating the question, no extra words, no blank lines, no markdown.
- Answer every question. If unsure, still pick the single most likely letter.

EXAMPLE REPLY (for 3 questions):
1: A
2: C
3: B
"""


In [ ]:
# Query the first batch. Allow enough tokens for all the numbered answers.
batch_result = query_model(
    prompts[0], MODEL_NAME, max_tokens=200, client=url, system_prompt=SYSTEM_PROMPT
)
print("Model's batched answer:\n")
print(batch_result["answer"])

### Parsing a batched answer

Now each line looks like `3: B`. We parse the whole block into a 
`{question_num: letter}` dict.

In [ ]:
def parse_batched_answers(text):
    """Parse 'N: X' lines into {N: 'X'} keeping only valid A-D letters."""
    answers = {}
    for line in text.strip().splitlines():
        m = re.match(r"\s*(\d+)\s*[:.]\s*([ABCD])", line.strip().upper())
        if m:
            answers[int(m.group(1))] = m.group(2)
    return answers


parsed = parse_batched_answers(batch_result["answer"])
print("Parsed:", parsed)

In [ ]:
def score_batch(parsed, answer_key, verbose=False):
    """Return accuracy of one parsed batch against its answer key."""
    correct = 0
    for qnum, gold in answer_key.items():
        got = parsed.get(qnum)
        if got == gold:
            correct += 1
        if verbose:
            mark = "OK" if got == gold else " X"
            print(f"{mark}\tQ{qnum}: correct={gold}, model={got}")
    return correct / len(answer_key)


acc = score_batch(parsed, answer_keys[0], verbose=True)
print(f"\nBatch accuracy: {acc:.1%}")

In [ ]:
def evaluate_dataset_batched(df, model_name, batch_size=10, system_prompt=SYSTEM_PROMPT,
                             max_tokens=200, pause=0.5):
    prompts, answer_keys = batch_questions(df, batch_size=batch_size)
    n_correct, n_total = 0, 0
    for prompt, key in tqdm(list(zip(prompts, answer_keys)), desc="Batches"):
        res = query_model(prompt, model_name, max_tokens=max_tokens,
                          client=url, system_prompt=system_prompt)
        time.sleep(pause)  # be gentle on the shared rate limit
        if res is None:
            continue
        parsed = parse_batched_answers(res["answer"])
        for qnum, gold in key.items():
            n_total += 1
            if parsed.get(qnum) == gold:
                n_correct += 1
    return n_correct / n_total if n_total else 0.0


overall_acc = evaluate_dataset_batched(df_full, MODEL_NAME, batch_size=BATCH_SIZE)
print(f"\nOverall accuracy on '{DATASET_NAME}' ({MODEL_NAME}): {overall_acc:.1%}")

## 5. Calibration

A model is **well calibrated** if, among answers it gives with 70% confidence,
about 70% are actually correct. We plot the model's **token probability** (x-axis)
against its **observed accuracy** (y-axis). A perfectly calibrated model sits on
the diagonal. We score the calibration with the **Expected Calibration Error (ECE)** — the average distance between confidence and accuracy, weighted by how many questions fall in each bin.

In [ ]:
from src.calibration_utils import (
    collect_confidences, 
    compute_calibration,
    plot_calibration,
    calibration_for
)

### 5a. One calibration plot

We run the full pipeline on one subject. Set `max_questions` to a small number
while testing, then `None` for the full subject.

In [ ]:
N_BINS = 10

cal_df = collect_confidences(df_full, MODEL_NAME, client=url, max_questions=50)
print(f"Collected {len(cal_df)} answered questions.")
print(f"Accuracy: {cal_df['is_correct'].mean():.1%}")

cal, ece = compute_calibration(cal_df, n_bins=N_BINS)


In [ ]:
title_text = f"Calibration - {MODEL_NAME}\n{DATASET_NAME} (N={len(cal_df)})"
plot_calibration(cal, ece, title=title_text)

**Reading the plot.** Points on the dashed diagonal mean well calibrated.
Points **below** the diagonal mean **overconfident** (the model claims more certainty
than its accuracy justifies) — the typical pattern for small models. `N=` shows how
many questions landed in each confidence bin; bins with few samples have large error
bars. 

### 5b. How calibration changes with model size

This is the main thing to observe: re-run the same pipeline for the **1B** and **8B**
models on the same subjects and compare. Do larger model produce more **accurate** and **calibrated** answers? 

In [ ]:
def calibration_for(dataset_name, model_name, client, n_bins=10, max_questions=None):
    """Full pipeline for one (subject, model): load -> query -> plot."""
    df = load_subject(dataset_name)
    res_df = collect_confidences(df, model_name, client=client, max_questions=max_questions)
    if len(res_df) == 0:
        print(f"No results for {dataset_name} / {model_name}.")
        return None
    cal, ece = compute_calibration(res_df, n_bins=n_bins)
    short = model_name.split("/")[-1]
    plot_calibration(cal, ece,
                     f"Calibration - {short}\n{dataset_name} "
                     f"(N={len(res_df)}, acc={res_df['is_correct'].mean():.0%})")
    return {"dataset": dataset_name, "model": model_name,
            "n": len(res_df), "accuracy": res_df["is_correct"].mean(), "ece": ece}

In [ ]:
# Compare model size on the two requested subjects.
# Keep max_questions modest to respect the shared rate limit; raise it for sharper plots.
SUBJECTS = ["college_biology", "college_physics"]
MODELS = [MODEL_1B, MODEL_8B]

summary = []
for subject in SUBJECTS:
    for model in MODELS:
        out = calibration_for(subject, model, client=url, n_bins=N_BINS)
        if out:
            summary.append(out)

pd.DataFrame(summary)

In the next notebook, you will see how to explore the model confidences on open ended questions. 